In [3]:
# h4_contextual_robustness.py
# Pooled fixed-effects test for H3 using outputs_h3/h3_summary_ALL.csv
# Produces: fig-h4-fe-tertile-effects.png and prints a LaTeX table.

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats

IN_CSV  = Path("outputs_h3") / "h3_summary_ALL.csv"
OUT_FIG = Path("outputs_h4") / "fig-h4-fe-tertile-effects.png"

df = pd.read_csv(IN_CSV)

# Build long panel: one row per (city, vehicle, orient, tertile, type)
rows = []
for _, r in df.iterrows():
    for tert, tm, rm in [
        ("Low",  "Low_T_median",  "Low_R_median"),
        ("Mid",  "Mid_T_median",  "Mid_R_median"),
        ("High", "High_T_median", "High_R_median"),
    ]:
        for typ, col in [("T", tm), ("R", rm)]:
            val = r[col]
            if pd.isna(val):
                continue
            rows.append({
                "City": r["City"],
                "Vehicle": r["Vehicle"],
                "Orientation": r["Orientation"],
                "BudgetPct": r["BudgetPct"],
                "Tertile": tert,
                "Type": typ,
                # convert to percentage points to match the paper
                "DeltaPct": 100.0 * float(val),
            })

long = pd.DataFrame(rows)
long["Treat"] = (long["Type"] == "T").astype(int)
long["Tertile"] = pd.Categorical(long["Tertile"], categories=["High","Mid","Low"], ordered=True)

# FE OLS with cluster-robust s.e. by city
# High RPE is the reference tertile
model = smf.ols(
    "DeltaPct ~ Treat + C(Tertile) + Treat:C(Tertile) + C(City) + C(Vehicle) + C(Orientation)",
    data=long,
).fit(cov_type="cluster", cov_kwds={"groups": long["City"]})

print(model.summary())

# Linear combinations: Targeted - Random by tertile
names = model.params.index.tolist()
cov = model.cov_params().values
params = model.params.values
name_to_idx = {n: i for i, n in enumerate(names)}

def eff_and_se(coeffs):
    w = np.zeros(len(names))
    for n, a in coeffs.items():
        if n not in name_to_idx:
            continue
        w[name_to_idx[n]] = a
    est = float(w @ params)
    se  = float(np.sqrt(w @ cov @ w))
    return est, se

eff_high = eff_and_se({"Treat": 1.0})
eff_mid  = eff_and_se({"Treat": 1.0, "Treat:C(Tertile)[T.Mid]": 1.0})
eff_low  = eff_and_se({"Treat": 1.0, "Treat:C(Tertile)[T.Low]": 1.0})

df_resid = model.df_resid

def ci95(est, se, df_):
    tcrit = stats.t.ppf(0.975, df_)
    return est - tcrit * se, est + tcrit * se

def pval_2s(est, se, df_):
    t = est / se
    return 2.0 * (1.0 - stats.t.cdf(abs(t), df_))

summary = pd.DataFrame({
    "Tertile": ["Low","Mid","High"],
    "FE effect (T-R)": [eff_low[0], eff_mid[0], eff_high[0]],
    "Std. Err.":       [eff_low[1], eff_mid[1], eff_high[1]],
    "p-value":         [pval_2s(*eff_low, df_resid),
                        pval_2s(*eff_mid, df_resid),
                        pval_2s(*eff_high, df_resid)],
    "CI 2.5%":         [ci95(*eff_low, df_resid)[0],
                        ci95(*eff_mid, df_resid)[0],
                        ci95(*eff_high, df_resid)[0]],
    "CI 97.5%":        [ci95(*eff_low, df_resid)[1],
                        ci95(*eff_mid, df_resid)[1],
                        ci95(*eff_high, df_resid)[1]],
})
print("\nPooled FE (Targeted - Random) by RPE tertile:\n")
print(summary.to_string(index=False, float_format=lambda x: f"{x:,.1f}"))

# Coefficient figure
plt.figure(figsize=(5.2, 3.6), dpi=150)
x = np.arange(3)
ests = summary["FE effect (T-R)"].values
ses  = summary["Std. Err."].values
plt.bar(x, ests, yerr=1.96*ses, capsize=4)
plt.xticks(x, summary["Tertile"])
plt.ylabel("FE estimate: Targeted minus Random (pp)")
plt.title("Pooled FE effect by RPE tertile (clustered by city)")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
OUT_FIG.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(OUT_FIG, bbox_inches="tight")
plt.close()

# Optional: emit a LaTeX table snippet
def latex_table(df, caption, label):
    cols = ["Tertile","FE effect (T-R)","Std. Err.","p-value","CI 2.5%","CI 97.5%"]
    df2 = df.copy()
    return (
        "\\begin{table}[ht!]\n\\centering\n\\small\n"
        f"\\caption{{{caption}}}\n\\label{{{label}}}\n"
        "\\begin{tabular}{lrrrrr}\n\\toprule\n"
        "Tertile & FE effect (T--R) & Std.\\ Err. & $p$--value & CI 2.5\\% & CI 97.5\\% \\\\\n\\midrule\n"
        + "\n".join(
            f"{r['Tertile']} & {r['FE effect (T-R)']:.1f} & {r['Std. Err.']:.1f} & {r['p-value']:.3f} & {r['CI 2.5%']:.1f} & {r['CI 97.5%']:.1f} \\\\"
            for _, r in df2[cols].iterrows()
        )
        + "\n\\bottomrule\n\\end{tabular}\n\\end{table}\n"
    )

print("\nLaTeX table snippet:\n")
print(latex_table(summary,
      "Fixed--effects estimates of the Targeted minus Random impact (percentage points) by RPE tertile; cluster--robust standard errors by city.",
      "tab:h4-fe-tertile"))

                            OLS Regression Results                            
Dep. Variable:               DeltaPct   R-squared:                       0.664
Model:                            OLS   Adj. R-squared:                  0.620
Method:                 Least Squares   F-statistic:                     189.0
Date:                Sat, 27 Sep 2025   Prob (F-statistic):           8.28e-05
Time:                        12:20:00   Log-Likelihood:                -474.09
No. Observations:                  96   AIC:                             972.2
Df Residuals:                      84   BIC:                             1003.
Df Model:                          11                                         
Covariance Type:              cluster                                         
                                              coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 11, but rank is 4
  warnings.warn('covariance of constraints does not have full '
